In [0]:
%run ../_shared/00_project_config

In [0]:
%run ../_shared/01_spark_utils

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import StructType


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

SOURCE_SYSTEM = "football-data.org"

BRONZE_TABLE = f"{BRONZE_SCHEMA}.football_data_org_teams"
SILVER_TABLE = f"{SILVER_SCHEMA}.football_data_org_teams"

# One row per competition, season and team.
BUSINESS_KEY = [
    "competition_id",
    "season_id",
    "team_id",
]

REQUIRED_SILVER_COLUMNS = [
    "competition_id",
    "competition_code",
    "league_code",
    "season_id",
    "season_start_year",
    "team_id",
    "team_name",
    "team_key",
]


# -----------------------------------------------------------------------------
# Read Bronze
# -----------------------------------------------------------------------------

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise ValueError(
        f"Required Bronze table does not exist: {BRONZE_TABLE}"
    )

bronze_df = spark.table(BRONZE_TABLE)


# -----------------------------------------------------------------------------
# Validate required top-level Bronze columns
# -----------------------------------------------------------------------------

REQUIRED_BRONZE_COLUMNS = [
    "competition",
    "season",
    "id",
    "name",
    "_league",
    "_ingested_at",
    "_source_file",
    "_source_system",
]

validate_columns_exist(
    bronze_df,
    REQUIRED_BRONZE_COLUMNS,
)

# -----------------------------------------------------------------------------
# Validate required nested Bronze fields
# -----------------------------------------------------------------------------

REQUIRED_NESTED_COLUMNS = [
    "competition.id",
    "competition.code",
    "season.id",
    "season.startDate",
]

missing_nested_columns = [
    path
    for path in REQUIRED_NESTED_COLUMNS
    if not field_exists(bronze_df, path)
]

if missing_nested_columns:
    raise ValueError(
        "Required nested Bronze columns are missing: "
        f"{missing_nested_columns}"
    )


# -----------------------------------------------------------------------------
# Clean top-level strings
# -----------------------------------------------------------------------------

bronze_df = clean_top_level_strings(bronze_df)


# -----------------------------------------------------------------------------
# Validate source identity
# -----------------------------------------------------------------------------

invalid_source_rows = (
    bronze_df
    .filter(
        F.col("_source_system").isNull()
        | (F.col("_source_system") != F.lit(SOURCE_SYSTEM))
    )
    .limit(1)
    .count()
)

if invalid_source_rows > 0:
    raise ValueError(
        f"Unexpected source system found in {BRONZE_TABLE}."
    )


# -----------------------------------------------------------------------------
# Select, rename and cast Silver columns
# -----------------------------------------------------------------------------

silver_df = (
    bronze_df
    .select(

        # ---------------------------------------------------------------------
        # Competition
        # ---------------------------------------------------------------------

        F.col("competition.id")
        .cast("int")
        .alias("competition_id"),

        clean_string("competition.code")
        .alias("competition_code"),

        optional_string(bronze_df, "competition.name")
        .alias("competition_name"),

        clean_string("_league")
        .alias("league_code"),


        # ---------------------------------------------------------------------
        # Season
        # ---------------------------------------------------------------------

        F.col("season.id")
        .cast("int")
        .alias("season_id"),

        F.col("season.startDate")
        .cast("date")
        .alias("season_start_date"),

        optional_col(bronze_df, "season.endDate", "date")
        .alias("season_end_date"),

        optional_col(bronze_df, "season.currentMatchday", "int")
        .alias("current_matchday"),


        # ---------------------------------------------------------------------
        # Team identifiers and names
        # ---------------------------------------------------------------------

        F.col("id")
        .cast("int")
        .alias("team_id"),

        clean_string("name")
        .alias("team_name"),

        optional_string(bronze_df, "shortName")
        .alias("team_short_name"),

        optional_string(bronze_df, "tla")
        .alias("team_tla"),

        optional_string(bronze_df, "crest")
        .alias("team_crest"),


        # ---------------------------------------------------------------------
        # Geographic information
        # ---------------------------------------------------------------------

        optional_col(bronze_df, "area.id", "int")
        .alias("area_id"),

        optional_string(bronze_df, "area.name")
        .alias("area_name"),

        optional_string(bronze_df, "area.code")
        .alias("area_code"),

        optional_string(bronze_df, "area.flag")
        .alias("area_flag"),


        # ---------------------------------------------------------------------
        # Club information
        # ---------------------------------------------------------------------

        optional_string(bronze_df, "address")
        .alias("address"),

        optional_string(bronze_df, "website")
        .alias("website"),

        optional_col(bronze_df, "founded", "int")
        .alias("founded_year"),

        optional_string(bronze_df, "clubColors")
        .alias("club_colors"),

        optional_string(bronze_df, "venue")
        .alias("venue"),


        # ---------------------------------------------------------------------
        # Source update information
        # ---------------------------------------------------------------------

        optional_col(bronze_df, "lastUpdated", "timestamp")
        .alias("last_updated"),


        # ---------------------------------------------------------------------
        # Bronze lineage
        # ---------------------------------------------------------------------

        F.col("_ingested_at")
        .cast("timestamp")
        .alias("_bronze_ingested_at"),

        F.col("_source_file")
        .cast("string")
        .alias("_bronze_source_file"),

        F.col("_source_system")
        .cast("string")
        .alias("_source_system"),
    )
)


# -----------------------------------------------------------------------------
# Normalize season
# -----------------------------------------------------------------------------

# The season starting year is the common representation
# used across the football-data.org Silver datasets.

silver_df = silver_df.withColumn(
    "season_start_year",
    F.year("season_start_date"),
)


# -----------------------------------------------------------------------------
# Validate business key before deduplication
# -----------------------------------------------------------------------------

validate_required_not_null(
    silver_df,
    BUSINESS_KEY,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Validate competition-to-league mapping
# -----------------------------------------------------------------------------

expected_league = map_values(
    "competition_code",
    FOOTBALL_DATA_ORG_COMPETITIONS,
)

invalid_mapping_rows = (
    silver_df
    .filter(
        expected_league.isNull()
        | F.col("league_code").isNull()
        | (F.col("league_code") != expected_league)
    )
    .limit(1)
    .count()
)

if invalid_mapping_rows > 0:
    raise ValueError(
        "Competition code and canonical league mapping "
        "are inconsistent."
    )


# -----------------------------------------------------------------------------
# Deduplicate repeated Bronze snapshots
# -----------------------------------------------------------------------------

# Bronze is append-oriented.
# Repeated ingestion can create multiple versions of a team.
# Keep the latest ingested record per competition, season and team.

silver_df = deduplicate_latest(
    silver_df,
    key_columns=BUSINESS_KEY,
    order_column="_bronze_ingested_at",
    tie_breaker_columns=[
        "last_updated",
        "_bronze_source_file",
    ],
)


# ---------------------------------------------------------------------------
# Resolve canonical team key
# ---------------------------------------------------------------------------

team_mappings_df = load_team_mappings_by_id(
    spark,
    SILVER_SCHEMA,
    SOURCE_SYSTEM,
)

silver_df = resolve_team_keys_by_id(
    silver_df,
    team_mappings_df,
    source_id_column="team_id",
    output_key_column="team_key",
)

print("Canonical team resolution completed.")



# -----------------------------------------------------------------------------
# Data quality: positive identifiers
# -----------------------------------------------------------------------------

invalid_id_rows = (
    silver_df
    .filter(
        (F.col("competition_id") <= 0)
        | (F.col("season_id") <= 0)
        | (F.col("team_id") <= 0)
    )
    .limit(1)
    .count()
)

if invalid_id_rows > 0:
    raise ValueError(
        "Invalid team data: identifiers must be positive."
    )


# -----------------------------------------------------------------------------
# Data quality: team names must not be blank
# -----------------------------------------------------------------------------

invalid_name_rows = (
    silver_df
    .filter(
        F.col("team_name").isNull()
        | (F.length(F.trim("team_name")) == 0)
    )
    .limit(1)
    .count()
)

if invalid_name_rows > 0:
    raise ValueError(
        "Invalid team data: missing or blank team names found."
    )


# -----------------------------------------------------------------------------
# Data quality: founded year
# -----------------------------------------------------------------------------

# Do not enforce an arbitrary minimum foundation year.
# A foundation year may also be NULL when unknown.

invalid_founded_rows = (
    silver_df
    .filter(
        F.col("founded_year").isNotNull()
        & (
            (F.col("founded_year") < 1800)
            | (F.col("founded_year") > F.year(F.current_date()))
        )
    )
    .limit(1)
    .count()
)

if invalid_founded_rows > 0:
    raise ValueError(
        "Invalid team data: founded_year is outside "
        "the expected historical range."
    )


# -----------------------------------------------------------------------------
# Data quality: uniqueness of team IDs within competition and season
# -----------------------------------------------------------------------------

duplicate_teams = (
    silver_df
    .groupBy(*BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_teams > 0:
    raise ValueError(
        "Duplicate team business keys found after deduplication."
    )


# -----------------------------------------------------------------------------
# Add Silver processing metadata
# -----------------------------------------------------------------------------

silver_df = silver_df.withColumn(
    "_silver_processed_at",
    F.current_timestamp(),
)


# -----------------------------------------------------------------------------
# Final Silver validation
# -----------------------------------------------------------------------------

validation_metrics = validate_required_not_null(
    silver_df,
    REQUIRED_SILVER_COLUMNS,
    fail_on_null=True,
)

if validation_metrics["row_count"] == 0:
    raise ValueError(
        "Silver teams dataset is empty. "
        "Refusing to overwrite the target table."
    )


# -----------------------------------------------------------------------------
# Write Silver
# -----------------------------------------------------------------------------

write_full_refresh(
    silver_df,
    SILVER_TABLE,
)


# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {validation_metrics['row_count']} "
    f"team records into {SILVER_TABLE}."
)
